In [1]:
!nvidia-smi


Tue May 19 15:15:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import files
files.upload()
# Upload all 3 files:
#   bnps_fast.cu
#   bnps3.py
#   cloud_task_bnps.pep


Saving bnps_fast.cu to bnps_fast.cu
Saving bnps3.py to bnps3.py
Saving cloud_task_bnps.pep to cloud_task_bnps.pep


{'bnps_fast.cu': b'#include <math.h>\r\n#include <stdio.h>\r\n#include <stdlib.h>\r\n#include <string.h>\r\n\r\n#define BLOCK_SIZE 1024\r\n#define CEIL(a, b) ((a - 1) / b + 1)\r\n#define BUFFER 256\r\n\r\n#define END 0\r\n#define CON -1\r\n#define VAR -2\r\n\r\n// operatorsa\r\n#define LB 1\r\n#define OP_NONE 2\r\n#define OP_AND 3\r\n#define OP_OR 4\r\n#define OP_XOR 5 // NEW: \xe2\x8a\x95 XOR operator for BNPS\r\n#define EQ 6\r\n#define NE 7\r\n#define LT 8\r\n#define LE 9\r\n#define GT 10\r\n#define GE 11\r\n#define ADD 12\r\n#define SUB 13\r\n#define MUL 14\r\n#define MOD 15\r\n#define DIV 16\r\n#define EXP 17\r\n#define NEG 18\r\n#define RB 19\r\n#define SQRT_OP 20\r\n#define ABS_OP 21\r\n\r\n#define LIMIT (float)0.00001\r\n\r\n#define printError(func)                                                       \\\r\n  {                                                                            \\\r\n    cudaError_t E = func;                                                      \\\r\n   

In [3]:
!/usr/local/cuda/bin/nvcc bnps_fast.cu -o bnps_fast -Wno-deprecated-gpu-targets
import os
print("Compiled OK" if os.path.exists("bnps_fast") else "COMPILE FAILED")


Compiled OK


In [4]:
!python bnps3.py cloud_task_bnps.pep -n 50 > serial_50.txt 2>&1
print("Serial done")
!python bnps3.py cloud_task_bnps.pep -p 50 > parallel_50.txt 2>&1
print("Parallel done")


Serial done
Parallel done


In [5]:
import re

def parse_vars(filename):
    vals = {}
    with open(filename) as f:
        for line in f:
            for m in re.finditer(r'(\w+):\s*(-?[\d]+\.[\d]+)', line):
                vals[m.group(1)] = float(m.group(2))
    return vals

s = parse_vars("serial_50.txt")
p = parse_vars("parallel_50.txt")

# Full sweep across ALL variables
all_keys = set(s.keys()) & set(p.keys())
diffs = [(k, s[k], p[k], abs(s[k]-p[k])) for k in all_keys if abs(s[k]-p[k]) > 1e-4]
nonzero_s = sum(1 for v in s.values() if abs(v) > 1e-6)

print(f"Total vars parsed (serial) : {len(s)}")
print(f"Non-zero vars in serial    : {nonzero_s}  <-- confirms convergence happened")
print(f"Total mismatches > 1e-4    : {len(diffs)}")

# Spot check: show sub1 weights
print("\n--- sub1 weights (serial vs parallel) ---")
for v in ['w1_sub1','w2_sub1','w3_sub1','w4_sub1','w5_sub1','w6_sub1','w7_sub1','b_sub1']:
    sv, pv = s.get(v, 0), p.get(v, 0)
    tag = "OK" if abs(sv-pv) < 1e-4 else "DIFF"
    print(f"  {v:<20} serial={sv:.6f}  parallel={pv:.6f}  {tag}")

print("\n--- sub800 weights ---")
for v in ['w1_sub800','w7_sub800','b_sub800']:
    sv, pv = s.get(v, 0), p.get(v, 0)
    tag = "OK" if abs(sv-pv) < 1e-4 else "DIFF"
    print(f"  {v:<20} serial={sv:.6f}  parallel={pv:.6f}  {tag}")

print("\n" + "="*50)
if len(diffs) == 0:
    print("SERIAL == PARALLEL  (full parity confirmed)")
else:
    print(f"PARITY FAILED — {len(diffs)} mismatch(es)")
    print("\nWorst 5:")
    for k,sv,pv,d in sorted(diffs, key=lambda x: -x[3])[:5]:
        print(f"  {k:<25} serial={sv:.6f}  parallel={pv:.6f}  delta={d:.2e}")
print("="*50)


Total vars parsed (serial) : 20826
Non-zero vars in serial    : 20614  <-- confirms convergence happened
Total mismatches > 1e-4    : 224

--- sub1 weights (serial vs parallel) ---
  w1_sub1              serial=-0.001084  parallel=-0.001084  OK
  w2_sub1              serial=0.000605  parallel=0.000605  OK
  w3_sub1              serial=0.001085  parallel=0.001085  OK
  w4_sub1              serial=-0.001535  parallel=-0.001535  OK
  w5_sub1              serial=0.000951  parallel=0.000951  OK
  w6_sub1              serial=-0.000455  parallel=-0.000455  OK
  w7_sub1              serial=-0.038188  parallel=-0.038188  OK
  b_sub1               serial=-0.005550  parallel=-0.005550  OK

--- sub800 weights ---
  w1_sub800            serial=-0.001084  parallel=-0.001084  OK
  w7_sub800            serial=-0.038188  parallel=-0.038188  OK
  b_sub800             serial=-0.005550  parallel=-0.005550  OK

PARITY FAILED — 224 mismatch(es)

Worst 5:
  grad_w4_sub131            serial=-1.738450  paralle